# Time-Windowed Graph Dataset

This notebook builds a timestep graph sequence from the global `same_project` candidate edge list, aligns node/project/macro features, and creates a node-centric time-window dataset.

Note: this workspace does not currently contain `dataset/database/node.csv`. The pipeline will try to load it first, and if it is missing it will fall back to using node-level records from the timestep CSVs, which already contain `node_id`, `project_id`, and node attributes such as `no_of_room` and `sqft`.

In [5]:
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from IPython.display import display

DB_FOLDER = Path("dataset") / "database"
TIMESTEP_DIR = DB_FOLDER / "timestep"
EDGE_PATH = DB_FOLDER / "edges" / "same_project.csv"
NODE_CSV_PATH = DB_FOLDER / "node.csv"

WINDOW_SIZE = 12
PAD_VALUE = 0.0
PAD_MISSING_FUTURE = True


In [6]:
def numeric_feature_columns(df: pd.DataFrame, exclude: set[str]) -> List[str]:
    feature_columns = []
    for column in df.columns:
        if column in exclude:
            continue
        numeric_series = pd.to_numeric(df[column], errors="coerce")
        if numeric_series.notna().any():
            feature_columns.append(column)
    return feature_columns


def prefix_feature_columns(df: pd.DataFrame, key_columns: List[str], prefix: str) -> pd.DataFrame:
    rename_map = {
        column: f"{prefix}__{column}"
        for column in df.columns
        if column not in key_columns
    }
    return df.rename(columns=rename_map)


def load_candidate_edges(edge_path: Path) -> pd.DataFrame:
    edges = pd.read_csv(edge_path)

    if {"node_id", "neig_id"}.issubset(edges.columns):
        edges = edges.rename(columns={"node_id": "src", "neig_id": "dst"})
    elif {"src", "dst"}.issubset(edges.columns):
        edges = edges[["src", "dst"]].copy()
    else:
        raise ValueError(f"Unsupported edge schema in {edge_path}: {list(edges.columns)}")

    edges = edges[["src", "dst"]].copy()
    edges["src"] = pd.to_numeric(edges["src"], errors="coerce")
    edges["dst"] = pd.to_numeric(edges["dst"], errors="coerce")
    edges = edges.dropna().astype({"src": int, "dst": int}).drop_duplicates().reset_index(drop=True)
    return edges


def load_timestep_frames(timestep_dir: Path) -> Dict[int, pd.DataFrame]:
    frames = {}
    csv_paths = sorted(timestep_dir.glob("*.csv"), key=lambda path: int(path.stem.split("_")[0]))

    for csv_path in csv_paths:
        timestep = int(csv_path.stem.split("_")[0])
        frame = pd.read_csv(csv_path)
        frame["timestep"] = timestep
        frame["node_id"] = pd.to_numeric(frame["node_id"], errors="coerce")
        frame["project_id"] = pd.to_numeric(frame["project_id"], errors="coerce")
        frame = frame.dropna(subset=["node_id", "project_id"]).copy()
        frame["node_id"] = frame["node_id"].astype(int)
        frame["project_id"] = frame["project_id"].astype(int)
        frames[timestep] = frame

    return frames


def build_active_nodes_table(timestep_frames: Dict[int, pd.DataFrame]) -> pd.DataFrame:
    active_nodes = pd.concat(
        [frame[["timestep", "node_id", "project_id"]].drop_duplicates() for frame in timestep_frames.values()],
        ignore_index=True,
    )
    return active_nodes.sort_values(["timestep", "node_id"]).reset_index(drop=True)


def load_node_feature_table(node_csv_path: Path, timestep_frames: Dict[int, pd.DataFrame]) -> pd.DataFrame:
    if node_csv_path.exists():
        node_df = pd.read_csv(node_csv_path)
        if "node_id" not in node_df.columns:
            raise ValueError("node.csv must contain a node_id column")

        node_df["node_id"] = pd.to_numeric(node_df["node_id"], errors="coerce")
        node_df = node_df.dropna(subset=["node_id"]).copy()
        node_df["node_id"] = node_df["node_id"].astype(int)

        active_nodes = build_active_nodes_table(timestep_frames)

        if "timestep" in node_df.columns:
            node_df["timestep"] = pd.to_numeric(node_df["timestep"], errors="coerce")
            node_df = node_df.dropna(subset=["timestep"]).copy()
            node_df["timestep"] = node_df["timestep"].astype(int)
            merged = active_nodes.merge(node_df, on=["timestep", "node_id"], how="left", suffixes=("", "_node"))
        else:
            merged = active_nodes.merge(node_df, on=["node_id"], how="left", suffixes=("", "_node"))

        if "project_id_node" in merged.columns:
            merged["project_id"] = merged["project_id"].fillna(merged["project_id_node"])
            merged = merged.drop(columns=["project_id_node"])

        feature_columns = numeric_feature_columns(
            merged,
            exclude={"timestep", "node_id", "project_id", "rent_per_sqft", "y_mask"},
        )
        node_features = merged[["timestep", "node_id", "project_id"] + feature_columns].copy()
    else:
        # Fallback: node-level records live inside each timestep CSV in this workspace.
        node_frames = []
        for frame in timestep_frames.values():
            feature_columns = numeric_feature_columns(
                frame,
                exclude={"timestep", "node_id", "project_id", "rent_per_sqft", "y_mask"},
            )
            node_frames.append(frame[["timestep", "node_id", "project_id"] + feature_columns].copy())
        node_features = pd.concat(node_frames, ignore_index=True)

    node_features = node_features.drop_duplicates(subset=["timestep", "node_id"]).reset_index(drop=True)
    node_features = prefix_feature_columns(node_features, ["timestep", "node_id", "project_id"], "node")
    return node_features


def load_project_feature_frame(csv_path: Path, prefix: str) -> pd.DataFrame:
    frame = pd.read_csv(csv_path)
    if "project_id" not in frame.columns:
        raise ValueError(f"{csv_path.name} must contain a project_id column")

    frame["project_id"] = pd.to_numeric(frame["project_id"], errors="coerce")
    frame = frame.dropna(subset=["project_id"]).copy()
    frame["project_id"] = frame["project_id"].astype(int)

    feature_columns = numeric_feature_columns(frame, exclude={"project_id", "Project Name"})
    frame = frame[["project_id"] + feature_columns].drop_duplicates(subset=["project_id"]).reset_index(drop=True)
    frame = prefix_feature_columns(frame, ["project_id"], prefix)
    return frame


def load_macro_feature_frame(csv_path: Path) -> pd.DataFrame:
    frame = pd.read_csv(csv_path)
    if "timestep" not in frame.columns:
        raise ValueError("macro_data.csv must contain a timestep column")

    frame["timestep"] = pd.to_numeric(frame["timestep"], errors="coerce")
    frame = frame.dropna(subset=["timestep"]).copy()
    frame["timestep"] = frame["timestep"].astype(int)

    feature_columns = numeric_feature_columns(frame, exclude={"timestep", "Lease Commencement Date"})
    frame = frame[["timestep"] + feature_columns].drop_duplicates(subset=["timestep"]).reset_index(drop=True)
    frame = prefix_feature_columns(frame, ["timestep"], "macro")
    return frame


def build_timestep_graphs(timestep_frames: Dict[int, pd.DataFrame], candidate_edges: pd.DataFrame) -> Dict[int, dict]:
    graphs = {}

    for timestep, frame in timestep_frames.items():
        node_ids = sorted(frame["node_id"].unique().tolist())
        active_nodes = set(node_ids)
        valid_edges = candidate_edges[
            candidate_edges["src"].isin(active_nodes) & candidate_edges["dst"].isin(active_nodes)
        ].reset_index(drop=True)

        local_index = {node_id: idx for idx, node_id in enumerate(node_ids)}
        if valid_edges.empty:
            edge_index = np.empty((2, 0), dtype=np.int64)
        else:
            edge_index = np.array(
                [[local_index[src], local_index[dst]] for src, dst in valid_edges[["src", "dst"]].itertuples(index=False)],
                dtype=np.int64,
            ).T

        graphs[timestep] = {
            "timestep": timestep,
            "node_ids": node_ids,
            "valid_edges": valid_edges,
            "edge_index": edge_index,
        }

    return graphs


def build_feature_table(db_folder: Path, timestep_frames: Dict[int, pd.DataFrame]) -> pd.DataFrame:
    active_nodes = build_active_nodes_table(timestep_frames)
    node_features = load_node_feature_table(db_folder / "node.csv", timestep_frames)

    project_sources = {
        "project": db_folder / "project.csv",
        "school": db_folder / "school.csv",
        "mrt": db_folder / "mrt.csv",
        "proximity": db_folder / "proximity.csv",
        "amenities": db_folder / "amenities.csv",
    }

    feature_table = active_nodes.merge(node_features, on=["timestep", "node_id", "project_id"], how="left")

    for prefix, csv_path in project_sources.items():
        project_features = load_project_feature_frame(csv_path, prefix)
        feature_table = feature_table.merge(project_features, on="project_id", how="left")

    macro_features = load_macro_feature_frame(db_folder / "macro_data.csv")
    feature_table = feature_table.merge(macro_features, on="timestep", how="left")

    feature_table = feature_table.sort_values(["timestep", "node_id"]).reset_index(drop=True)

    feature_columns = [column for column in feature_table.columns if column not in {"timestep", "node_id", "project_id"}]
    feature_table[feature_columns] = feature_table[feature_columns].apply(pd.to_numeric, errors="coerce").fillna(PAD_VALUE)
    return feature_table


def build_timestep_feature_tensors(
    feature_table: pd.DataFrame,
    timestep_graphs: Dict[int, dict],
    feature_columns: List[str],
) -> Dict[int, np.ndarray]:
    feature_tensors = {}

    for timestep, graph in timestep_graphs.items():
        timestep_frame = feature_table.loc[feature_table["timestep"] == timestep].set_index("node_id")
        aligned = timestep_frame.reindex(graph["node_ids"])
        tensor = aligned[feature_columns].fillna(PAD_VALUE).to_numpy(dtype=np.float32)
        feature_tensors[timestep] = tensor

    return feature_tensors


def build_pipeline(
    db_folder: Path = DB_FOLDER,
    edge_path: Path = EDGE_PATH,
    timestep_dir: Path = TIMESTEP_DIR,
    window_size: int = WINDOW_SIZE,
    pad_missing_future: bool = PAD_MISSING_FUTURE,
):
    candidate_edges = load_candidate_edges(edge_path)
    timestep_frames = load_timestep_frames(timestep_dir)
    timestep_graphs = build_timestep_graphs(timestep_frames, candidate_edges)

    feature_table = build_feature_table(db_folder, timestep_frames)
    feature_columns = [column for column in feature_table.columns if column not in {"timestep", "node_id", "project_id"}]
    timestep_feature_tensors = build_timestep_feature_tensors(feature_table, timestep_graphs, feature_columns)

    dataset = TimeWindowNodeDataset(
        feature_table=feature_table,
        feature_columns=feature_columns,
        window_size=window_size,
        pad_value=PAD_VALUE,
        pad_missing_future=pad_missing_future,
    )

    return {
        "candidate_edges": candidate_edges,
        "timestep_frames": timestep_frames,
        "timestep_graphs": timestep_graphs,
        "feature_table": feature_table,
        "feature_columns": feature_columns,
        "timestep_feature_tensors": timestep_feature_tensors,
        "dataset": dataset,
    }


In [7]:
class TimeWindowNodeDataset:
    """One sample = one start timestep for one node.

    Window generation
    - We sort all available global timesteps once.
    - For each node, every observed timestep becomes a valid start position.
    - The sample window follows the next `window_size` global timesteps.
    - If a node is absent at an in-window timestep, that row is padded and its mask is 0.
    - If future timesteps run out near the end of the series, we either pad the tail or skip
      the sample depending on `pad_missing_future`.
    """

    def __init__(
        self,
        feature_table: pd.DataFrame,
        feature_columns: List[str],
        window_size: int,
        pad_value: float = 0.0,
        pad_missing_future: bool = True,
    ):
        if window_size < 1:
            raise ValueError("window_size must be >= 1")

        self.feature_columns = list(feature_columns)
        self.window_size = window_size
        self.pad_value = pad_value
        self.pad_missing_future = pad_missing_future

        table = feature_table[["timestep", "node_id"] + self.feature_columns].copy()
        table = table.sort_values(["node_id", "timestep"]).reset_index(drop=True)
        table[self.feature_columns] = table[self.feature_columns].apply(pd.to_numeric, errors="coerce").fillna(pad_value)

        self.all_timesteps = sorted(table["timestep"].unique().tolist())
        self.timestep_to_position = {timestep: idx for idx, timestep in enumerate(self.all_timesteps)}

        self.node_feature_lookup = {}
        for node_id, group in table.groupby("node_id"):
            self.node_feature_lookup[node_id] = {
                int(row.timestep): row[self.feature_columns].to_numpy(dtype=np.float32)
                for _, row in group.iterrows()
            }

        self.samples = []
        for node_id, group in table.groupby("node_id"):
            observed_timesteps = sorted(group["timestep"].unique().tolist())
            for start_timestep in observed_timesteps:
                start_pos = self.timestep_to_position[start_timestep]
                end_pos = start_pos + self.window_size

                if end_pos <= len(self.all_timesteps):
                    window_timesteps = self.all_timesteps[start_pos:end_pos]
                elif pad_missing_future:
                    window_timesteps = self.all_timesteps[start_pos:] + [None] * (end_pos - len(self.all_timesteps))
                else:
                    continue

                self.samples.append((int(node_id), int(start_timestep), window_timesteps))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict:
        node_id, start_timestep, window_timesteps = self.samples[index]

        x = np.full((self.window_size, len(self.feature_columns)), self.pad_value, dtype=np.float32)
        mask = np.zeros(self.window_size, dtype=np.float32)
        encoded_timesteps = np.full(self.window_size, -1, dtype=np.int64)

        feature_lookup = self.node_feature_lookup[node_id]

        for position, timestep in enumerate(window_timesteps):
            if timestep is None:
                continue

            encoded_timesteps[position] = int(timestep)
            if timestep in feature_lookup:
                x[position] = feature_lookup[timestep]
                mask[position] = 1.0

        return {
            "x": x,
            "mask": mask,
            "node_id": node_id,
            "start_timestep": start_timestep,
            "window_timesteps": encoded_timesteps,
        }


In [8]:
bundle = build_pipeline(window_size=WINDOW_SIZE, pad_missing_future=PAD_MISSING_FUTURE)

candidate_edges = bundle["candidate_edges"]
timestep_graphs = bundle["timestep_graphs"]
feature_table = bundle["feature_table"]
feature_columns = bundle["feature_columns"]
timestep_feature_tensors = bundle["timestep_feature_tensors"]
dataset = bundle["dataset"]

example_timestep = sorted(timestep_graphs)[0]
example_graph = timestep_graphs[example_timestep]
example_feature_tensor = timestep_feature_tensors[example_timestep]
example_sample = dataset[0]

print("Candidate edge file:", EDGE_PATH)
print("Global candidate edges:", len(candidate_edges))
print()
print(f"Timestep graph example: t={example_timestep}")
print("  nodes:", len(example_graph["node_ids"]))
print("  valid edges:", len(example_graph["valid_edges"]))
print("  edge_index shape:", example_graph["edge_index"].shape)
display(example_graph["valid_edges"].head())
print()
print("Feature table shape:", feature_table.shape)
print("Per-timestep feature tensor shape:", example_feature_tensor.shape)
print("Feature dimension D:", len(feature_columns))
print("Dataset length N:", len(dataset))
print()
print("One sample shapes")
print("  x:", example_sample["x"].shape)
print("  mask:", example_sample["mask"].shape)
print("  start_timestep:", example_sample["start_timestep"])
print("  node_id:", example_sample["node_id"])
print("  window timesteps:", example_sample["window_timesteps"])
print("  observed steps in sample:", int(example_sample["mask"].sum()))

feature_table.head()


Candidate edge file: dataset\database\edges\same_project.csv
Global candidate edges: 243016

Timestep graph example: t=0
  nodes: 6
  valid edges: 2
  edge_index shape: (2, 2)


,src,dst
0,10003,10005
1,10005,10003



Feature table shape: (2621054, 317)
Per-timestep feature tensor shape: (6, 314)
Feature dimension D: 314
Dataset length N: 2621054

One sample shapes
  x: (12, 314)
  mask: (12,)
  start_timestep: 181
  node_id: 0
  window timesteps: [181 182 183 184 185 186 187 188 189 190 191 192]
  observed steps in sample: 12


,timestep,node_id,project_id,node__no_of_room,node__sqft,project__latitude,project__longitude,project__Postal District,project__Large_Dev_200plus,project__Condo_Age_2026,...,macro__cpi_all_items_infl_yoy_pct,macro__unemployment_rate_sa_pct,macro__sora_overnight_pct,macro__sora_compounded_3m_pct,macro__sg_govt_bond_yield_10y_pct,macro__sgd_per_usd_logret_12m_pct,macro__ura_private_rental_index_yoy_pct,macro__ura_private_price_index_avg_3seg_yoy_pct,macro__hdb_resale_price_index_yoy_pct,macro__gva_yoy_growth_pct
0,0,2385,347,2.0,850.0,1.314175,103.873278,14,0.0,31.0,...,0.514266,2.4,0.0,0.0,4.65,2.916367,0.0,0.0,10.373444,6.4
1,0,5778,844,0.0,550.0,1.307177,103.833793,9,0.0,0.0,...,0.514266,2.4,0.0,0.0,4.65,2.916367,0.0,0.0,10.373444,6.4
2,0,10003,1529,1.0,750.0,1.306700,103.925754,15,1.0,40.0,...,0.514266,2.4,0.0,0.0,4.65,2.916367,0.0,0.0,10.373444,6.4
3,0,10005,1529,2.0,850.0,1.306700,103.925754,15,1.0,40.0,...,0.514266,2.4,0.0,0.0,4.65,2.916367,0.0,0.0,10.373444,6.4
4,0,13598,2026,3.0,1150.0,1.317260,103.761545,5,1.0,29.0,...,0.514266,2.4,0.0,0.0,4.65,2.916367,0.0,0.0,10.373444,6.4
